# Project: Kalman Filter – track a cyclist through Aalborg 🚲📡
Your phone's GPS jumps around by ~8 m. A **Kalman filter** combines a motion model ("bikes move smoothly") with noisy measurements, and the result is better than either alone. It is Bayesian inference with Gaussians, run once per time step: predict (prior) → update with the measurement (posterior).

Topic: [[Bayesian Inference]], [[Probabilistic Graphical Models]] · guide note: [[Project - Kalman Filter Tracker]]

**How this works:** cells marked **TODO** are yours. Run the ✅ check under each one; later cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*. Data is downloaded once into `./data` (git-ignored).

In [ ]:
import numpy as np, matplotlib.pyplot as plt
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

rng = np.random.default_rng(0); dt, T = 1.0, 300
# true path: a cyclist with smoothly changing velocity (m, m/s); state = [x, y, vx, vy]
true = np.zeros((T, 4)); true[0] = [0, 0, 4, 1]
for t in range(1, T):
    acc = rng.normal(0, 0.15, 2)
    true[t, 2:] = true[t - 1, 2:] + acc * dt
    true[t, :2] = true[t - 1, :2] + true[t - 1, 2:] * dt
GPS_SD = 8.0
z = true[:, :2] + rng.normal(0, GPS_SD, (T, 2))           # noisy GPS fixes
F = np.array([[1, 0, dt, 0], [0, 1, 0, dt], [0, 0, 1, 0], [0, 0, 0, 1]], float)   # constant-velocity model
H = np.array([[1, 0, 0, 0], [0, 1, 0, 0]], float)                                   # we observe position only
q = 0.15 ** 2; Q = q * np.array([[dt**4 / 4, 0, dt**3 / 2, 0], [0, dt**4 / 4, 0, dt**3 / 2], [dt**3 / 2, 0, dt**2, 0], [0, dt**3 / 2, 0, dt**2]])
R = GPS_SD ** 2 * np.eye(2)
plt.plot(*true[:, :2].T, "k", label="true path"); plt.scatter(*z.T, s=4, c="C1", label="GPS"); plt.axis("equal"); plt.legend(); plt.show()

## 1. Predict (the prior)
Push the belief through the motion model: $x \leftarrow Fx$, and the uncertainty grows: $P \leftarrow FPF^\top + Q$.

In [ ]:
def predict(x, P, F, Q):
    # TODO 1: prior mean and covariance
    raise NotImplementedError  # TODO 1

## 2. Update (the posterior)
Innovation covariance $S = HPH^\top + R$, **Kalman gain** $K = PH^\top S^{-1}$, then $x \leftarrow x + K(z - Hx)$ and $P \leftarrow (I - KH)P$. $K$ automatically weighs the measurement against the prediction by their uncertainties.

In [ ]:
def update(x, P, z, H, R):
    # TODO 2: Kalman gain, posterior mean and covariance
    raise NotImplementedError  # TODO 2

In [ ]:
def _t_kf():
    # 1-D sanity check: prior N(0, 4), measurement 2 with variance 4 → posterior N(1, 2) (precision-weighted average)
    x, P = update(np.array([0.0]), np.array([[4.0]]), np.array([2.0]), np.eye(1), np.array([[4.0]]))
    xp, Pp = predict(np.array([1.0, 2.0]), np.eye(2), np.array([[1, 1], [0, 1.0]]), np.zeros((2, 2)))
    return np.allclose(x, [1.0]) and np.allclose(P, [[2.0]]) and np.allclose(xp, [3, 2]) and np.allclose(Pp, [[2, 1], [1, 1]])
check("kalman steps", _t_kf, "1-D: N(0,4) prior + measurement 2 (var 4) → N(1,2). Predict: Fx and FPFᵀ+Q.")

In [ ]:
rmse = lambda a, b: float(np.sqrt(np.mean(np.sum((a - b) ** 2, 1))))
if ready("kalman steps"):
    x, P = np.array([z[0, 0], z[0, 1], 0, 0]), np.diag([GPS_SD**2, GPS_SD**2, 25, 25]); est, sd = [], []
    for t in range(T):
        if t > 0: x, P = predict(x, P, F, Q)
        x, P = update(x, P, z[t], H, R); est.append(x.copy()); sd.append(np.sqrt(P[0, 0]))
    est = np.array(est)
    r_gps, r_kf = rmse(z, true[:, :2]), rmse(est[:, :2], true[:, :2])
    print(f"position error: raw GPS {r_gps:.2f} m  →  Kalman {r_kf:.2f} m;  speed error {rmse(est[:, 2:], true[:, 2:]):.2f} m/s (never measured!)")
    plt.plot(*true[:, :2].T, "k", label="true"); plt.scatter(*z.T, s=4, c="C1", alpha=.5, label="GPS"); plt.plot(*est[:, :2].T, "C0", label="Kalman")
    plt.axis("equal"); plt.legend(); plt.show()
    check("filter beats GPS", lambda: r_kf < 0.6 * r_gps, "The filtered track should be at least 40 % more accurate than raw GPS.")

<details><summary>▶ Solution</summary>

```python
def predict(x, P, F, Q):
    x = F @ x
    P = F @ P @ F.T + Q
    return x, P
```

```python
def update(x, P, z, H, R):
    S = H @ P @ H.T + R
    K = P @ H.T @ np.linalg.inv(S)
    x = x + K @ (z - H @ x)
    P = (np.eye(len(x)) - K @ H) @ P
    return x, P
```
</details>

## Stretch goals
- Mis-specify the noise: set `R` 10× too small or too large. What happens to the track, and why?
- Drop GPS for 30 seconds (a tunnel): skip `update` and watch the uncertainty `sd` grow.
- Add an RTS **smoother** (a backward pass): the offline version is even better. Same idea as forward–backward in HMMs.